# Observable and Hamiltonian Translation Workflow

## Problem

Circuit translation is only one part of SDK migration. Variational algorithms, chemistry workflows, and QAOA-style workloads also need observables or Hamiltonians to move between SDKs.

## Translation Scope

This workflow uses the first supported non-circuit semantic layer: weighted sums of Pauli `I`, `X`, `Y`, and `Z` products. The same representation covers single observables and Pauli Hamiltonians.

## Variables and Parameters

- `hamiltonian_source`: neutral `pauli-json` source for the weighted Pauli terms.
- `from_format`: source Hamiltonian format, here `pauli-json`.
- `to_format`: target SDK or neutral format used by each translation.
- `TARGETS`: all local/free Hamiltonian output targets covered by this notebook.
- `verify`: semantic verification mode, here canonical Pauli-term verification.
- `ARTIFACT_DIR`: notebook artifact directory for translated source and report JSON.


## Setup

Use the public observable/Hamiltonian translation API and the same notebook artifact directory convention as the other tutorials.

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from quantum_backend_bench.core.observable_translate import (
    HAMILTONIAN_OUTPUT_FORMATS,
    hamiltonian_check_report,
    hamiltonian_translation_report,
    import_hamiltonian_source,
    translate_hamiltonian_source,
    translation_capability_rows,
)
from quantum_backend_bench.utils.notebook import notebook_artifact_dir, verification_frame

ARTIFACT_DIR = notebook_artifact_dir()
TARGETS = ["qiskit_aer", "cirq", "pennylane", "braket_local", "pauli-json"]


def source_preview(title, source, max_lines=32):
    lines = source.strip().splitlines()
    print(title)
    print("-" * len(title))
    for number, line in enumerate(lines[:max_lines], start=1):
        print(f"{number:>2}: {line}")
    if len(lines) > max_lines:
        print(f"... {len(lines) - max_lines} more lines")


def translation_summary(results):
    rows = []
    for target, result in results.items():
        rows.append(
            {
                "target": target,
                "verified": result.verification.passed if result.verification else None,
                "details": result.verification.details if result.verification else "not checked",
                "source_lines": len(result.source.strip().splitlines()),
            }
        )
    return pd.DataFrame(rows)


def artifact_frame(rows):
    return pd.DataFrame(rows, columns=["artifact", "target", "path"])

## Source Hamiltonian

Start from neutral `pauli-json`. This keeps the example independent of any one SDK while still representing the same Pauli terms each target SDK can express.

In [ ]:
hamiltonian_source = json.dumps(
    {
        "n_qubits": 2,
        "terms": [
            {"coefficient": 0.5, "paulis": {"0": "Z", "1": "Z"}},
            {"coefficient": -1.25, "paulis": {"0": "X"}},
        ],
    },
    indent=2,
    sort_keys=True,
)

source_preview("Input pauli-json Hamiltonian", hamiltonian_source)

## Preflight Inspection

Import the source into the neutral Pauli Hamiltonian model and show the supported output formats.

In [ ]:
hamiltonian, detected_format = import_hamiltonian_source(
    hamiltonian_source,
    from_format="pauli-json",
)
check_report = hamiltonian_check_report(
    hamiltonian,
    detected_format,
    source_path="inline:hamiltonian_source",
)

display(
    pd.DataFrame(
        [
            {"field": "input format", "value": check_report["input_format"]},
            {"field": "qubits", "value": check_report["n_qubits"]},
            {"field": "terms", "value": check_report["term_count"]},
            {"field": "pauli counts", "value": check_report["pauli_counts"]},
            {"field": "supported outputs", "value": ", ".join(HAMILTONIAN_OUTPUT_FORMATS)},
        ]
    )
)

## Hamiltonian Coefficient Plot

Plot the weighted Pauli terms before translation. This gives a quick visual check of term signs and relative magnitudes independent of any SDK-specific object syntax.


In [ ]:
term_rows = []
for term in hamiltonian.terms:
    label = " ".join(f"${pauli}_{wire}$" for wire, pauli in term.paulis) or "$I$"
    term_rows.append({"term": label, "coefficient": term.coefficient})

term_frame = pd.DataFrame(term_rows)
display(term_frame)

fig, ax = plt.subplots(figsize=(6.5, 3.5))
colors = ["#2a9d8f" if value >= 0 else "#e76f51" for value in term_frame["coefficient"]]
ax.bar(term_frame["term"], term_frame["coefficient"], color=colors)
ax.axhline(0, color="#555555", linewidth=0.8)
ax.set_title("Pauli Hamiltonian coefficients")
ax.set_xlabel("Pauli term")
ax.set_ylabel("coefficient")
plt.tight_layout()
plt.show()

## Translate to All Local SDK Targets

Translate to Qiskit Aer, Cirq, PennyLane, Braket LocalSimulator, and back to neutral JSON. Canonical verification reimports each generated target and compares qubit-indexed Pauli terms.

In [ ]:
translation_results = {
    target: translate_hamiltonian_source(
        hamiltonian_source,
        from_format="pauli-json",
        to_format=target,
        verify="canonical",
    )
    for target in TARGETS
}

display(translation_summary(translation_results))

## Translated Source Previews

Each SDK has its own idiomatic object: Qiskit `SparsePauliOp`, Cirq Pauli expressions, PennyLane `qml.Hamiltonian`, and Braket observable terms with explicit targets.

In [ ]:
for target, result in translation_results.items():
    source_preview(f"{target} Hamiltonian source", result.source)
    print()

## Save Artifacts

Write one source artifact per target and one combined report for migration auditing.

In [ ]:
artifact_rows = []
combined_report = []
for target, result in translation_results.items():
    extension = ".json" if target == "pauli-json" else ".py"
    source_path = ARTIFACT_DIR / f"hamiltonian_pauli_json_to_{target}{extension}"
    source_path.write_text(result.source, encoding="utf-8")
    combined_report.append(
        hamiltonian_translation_report(
            result,
            source_path="inline:hamiltonian_source",
            from_format="pauli-json",
            to_format=target,
        )
    )
    artifact_rows.append(
        {"artifact": "translated Hamiltonian", "target": target, "path": str(source_path)}
    )

report_path = ARTIFACT_DIR / "hamiltonian_pauli_json_all_targets_report.json"
report_path.write_text(
    json.dumps(combined_report, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)
artifact_rows.append({"artifact": "combined report", "target": "all", "path": str(report_path)})

artifact_frame(artifact_rows)

## Translation Audit

The audit table makes the current translation surface explicit. Circuit translation and Pauli Hamiltonian translation are available now; broader SDK function translation remains intentionally unsupported until each semantic layer has safe verification.

In [ ]:
display(pd.DataFrame(translation_capability_rows()))

## Verification Summary

All targets should pass canonical Pauli-term verification.

In [ ]:
checks = [
    {
        "check": f"{target} canonical verification",
        "value": result.verification.details if result.verification else "not checked",
        "expected": "Canonical Pauli-term verification passed.",
        "passed": result.verification.passed if result.verification else False,
    }
    for target, result in translation_results.items()
]

verification_frame(checks)